# 02 – Modelling: predicting the category from the description

Input: `transaction_description` · Target: `category` · Main metric: macro-F1
Two evaluations: random split (merchants already seen) and merchant split (new merchants).

In [1]:
import pandas as pd

clean = pd.read_parquet("../data/clean_sample.parquet")
print(clean.shape)
clean.head()

(200000, 3)


,transaction_description,merchant,category
0,Exxon - CANADA Store,exxon,Transportation
1,Dunkin' Donuts - USA Branch,dunkin' donuts,Food & Dining
2,Zoo - INDIA Store - Night,zoo,Entertainment & Recreation
3,Lyft - UK Branch,lyft,Transportation
4,Ulta #9601 Store (Cash),ulta,Shopping & Retail


In [2]:
# Random Split
from sklearn.model_selection import train_test_split

train_r, test_r = train_test_split(
    clean, test_size=0.2, random_state=42, stratify=clean["category"]
)
print(len(train_r), len(test_r))

160000 40000


In [3]:
# Merchant split
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(clean, groups=clean["merchant"]))
train_m, test_m = clean.iloc[train_idx], clean.iloc[test_idx]
print(len(train_m), len(test_m))

160485 39515


In [4]:
seen_r = test_r["merchant"].isin(train_r["merchant"]).mean()
seen_m = test_m["merchant"].isin(train_m["merchant"]).mean()
print(f"Test rows with a merchant seen in training - random: {seen_r:.1%}, merchant: {seen_m:.1%}")

Test rows with a merchant seen in training - random: 100.0%, merchant: 0.0%


In [5]:
from sklearn.metrics import f1_score, accuracy_score

def evaluate(model, train, test):
    model.fit(train["transaction_description"], train["category"])
    pred = model.predict(test["transaction_description"])
    return {
        "accuracy": accuracy_score(test["category"], pred),
        "macro_f1": f1_score(test["category"], pred, average="macro"),
    }

In [6]:
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

models = {
    "dummy": make_pipeline(TfidfVectorizer(), DummyClassifier(strategy="most_frequent")),
    "tfidf_word_lr": make_pipeline(
        TfidfVectorizer(analyzer="word", ngram_range=(1, 2)),
        LogisticRegression(max_iter=1000),
    ),
    "tfidf_char_lr": make_pipeline(
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True),
        LogisticRegression(max_iter=1000),
    ),
}

In [7]:
results = []
for name, model in models.items():
    for split_name, (tr, te) in {"random": (train_r, test_r), "merchant": (train_m, test_m)}.items():
        scores = evaluate(model, tr, te)
        results.append({"model": name, "split": split_name, **scores})

results_df = pd.DataFrame(results).pivot(index="model", columns="split", values="macro_f1").round(3)
results_df

split,merchant,random
model,,
dummy,0.011,0.018
tfidf_char_lr,0.367,0.986
tfidf_word_lr,0.238,0.986


### First results (macro-F1)

| Model | Random split | Merchant split |
|---|---|---|
| Dummy (most frequent) | 0.018 | 0.011 |
| Word TF-IDF + LR | 0.986 | 0.238 |
| Char TF-IDF + LR | 0.986 | 0.367 |

- **Random split: 0.986** for both TF-IDF models, at the ~98.5% ceiling set by the 7 ambiguous merchants.
  Every test merchant was also in training (100%), so this score measures **memorising ~360 merchant names**,
  not understanding.
- **Merchant split: 0.367 (char) / 0.238 (word).** On merchants it has never seen, the model mostly fails.
  It is still well above chance (~0.10 for 10 categories), so it does learn some real clues.
- **Why so low:** the 4.5M rows contain only ~480 merchants. Repeating each one with random suffixes adds
  no new information, so the model effectively learns from a few hundred examples.
- **Characters beat words on new merchants:** 2–4 letter chunks match parts of unseen names
  ("pizzeria" shares "pizz" with "pizza"), while word features need an exact known word.
- **Conclusion:** a random split would have reported 98.6% and been misleading. The merchant split is the
  honest estimate of how the model will do on a real user's new merchants.

In [8]:
char_model = models["tfidf_char_lr"]
char_model.fit(train_m["transaction_description"], train_m["category"])

test_m = test_m.copy()
test_m["pred"] = char_model.predict(test_m["transaction_description"])
test_m["correct"] = test_m["pred"] == test_m["category"]

per_merchant = (test_m.groupby("merchant")
                .agg(category=("category", "first"),
                     top_pred=("pred", lambda p: p.mode()[0]),
                     accuracy=("correct", "mean"))
                .sort_values("accuracy"))

print(per_merchant.head(20))   # worst merchants
print(per_merchant.tail(20))   # best merchants

                                  category                    top_pred  \
merchant                                                                 
ambulance               Government & Legal  Entertainment & Recreation   
arcade          Entertainment & Recreation           Shopping & Retail   
atf                     Government & Legal        Utilities & Services   
carnival        Entertainment & Recreation        Healthcare & Medical   
burlington               Shopping & Retail              Transportation   
bonus                               Income               Food & Dining   
bloomingdale's           Shopping & Retail               Food & Dining   
bart                        Transportation           Shopping & Retail   
comerica                Financial Services        Utilities & Services   
checkers                     Food & Dining         Charity & Donations   
dollar                      Transportation           Shopping & Retail   
dart                        Transporta

## Error analysis (merchant split, char TF-IDF + LR)

**What it gets right:** names that share a word with merchants seen in training
(`la fitness`, `24 hour fitness`, `pharmacy`, `clinic`, `investment bank`, `electric company`).

**What it gets wrong:**
- **Brands and abbreviations with no clue in the name:** `chase`, `comerica`, `netflix`
- **Generic words that appear in only one merchant:** `festival`, `carnival`, `arcade`, `insurance`.
  The word has meaning, but the model never saw it during training, so knowing what words mean would help.

**All or nothing:** each merchant is either fully right or fully wrong, because the suffixes
(`- UK`, `(Card)`) are random noise and the model decides from the name alone. The merchant-split
score is therefore based on ~90 test merchants, so a single split is unreliable -> GroupKFold next.

## Reliable merchant-split score (GroupKFold)
One merchant split tests only ~90 merchants, so the score depends on which merchants land in test.
GroupKFold splits the merchants into 5 groups and tests each one once, so every merchant is tested.

In [9]:
from sklearn.model_selection import GroupKFold, cross_val_score

gkf = GroupKFold(n_splits=5)

cv_results = {}
for name in ["tfidf_word_lr", "tfidf_char_lr"]:
    scores = cross_val_score(
        models[name],
        clean["transaction_description"],
        clean["category"],
        groups=clean["merchant"],
        cv=gkf,
        scoring="f1_macro",
        n_jobs=-1,
    )
    cv_results[name] = scores
    print(f"{name}: {scores.mean():.3f} ± {scores.std():.3f}   folds: {scores.round(3)}")

tfidf_word_lr: 0.323 ± 0.018   folds: [0.309 0.325 0.318 0.307 0.357]
tfidf_char_lr: 0.386 ± 0.032   folds: [0.387 0.373 0.385 0.343 0.441]


### Merchant-split score with GroupKFold (5 folds, macro-F1)

| Model | Mean ± std | Folds |
|---|---|---|
| Word TF-IDF + LR | 0.323 ± 0.018 | 0.309, 0.325, 0.318, 0.307, 0.357 |
| Char TF-IDF + LR | 0.386 ± 0.032 | 0.387, 0.373, 0.385, 0.343, 0.441 |

- The single split under-estimated the word model (0.238 vs 0.323): which ~90 merchants land in
  test changes the score by up to ~0.1, so one split is not reliable.
- Char beats word in every fold, so the advantage of character n-grams is consistent.
- These GroupKFold means are the reference scores for all further models.

## Sentence embeddings + logistic regression
TF-IDF only knows the letters it saw in this dataset. A sentence-embedding model (`all-MiniLM-L6-v2`)
was pretrained on large amounts of text and turns a description into 384 numbers that capture its meaning.
Question: does this knowledge help on new merchants (brands, rare words like "festival")?
The logistic regression stays the same; only the text -> numbers step changes.

In [10]:
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

emb_path = Path("../data/X_emb.npy")

if emb_path.exists():
    X_emb = np.load(emb_path)
else:
    encoder = SentenceTransformer("all-MiniLM-L6-v2")
    unique_desc = clean["transaction_description"].unique()
    emb = encoder.encode(list(unique_desc), batch_size=256, show_progress_bar=True)
    lookup = dict(zip(unique_desc, emb))
    X_emb = np.vstack(clean["transaction_description"].map(lookup).to_numpy())
    np.save(emb_path, X_emb)

print(X_emb.shape)   # expected: (200000, 384)

(200000, 384)


In [11]:
emb_scores = cross_val_score(
    LogisticRegression(max_iter=1000),
    X_emb,
    clean["category"],
    groups=clean["merchant"],
    cv=gkf,
    scoring="f1_macro",
    n_jobs=-1,
)
cv_results["embeddings_lr"] = emb_scores
print(f"embeddings_lr: {emb_scores.mean():.3f} ± {emb_scores.std():.3f}   folds: {emb_scores.round(3)}")

embeddings_lr: 0.621 ± 0.032   folds: [0.635 0.672 0.598 0.622 0.579]


### Sentence embeddings (GroupKFold, macro-F1)

| Model | Merchant split |
|---|---|
| Word TF-IDF + LR | 0.323 ± 0.018 |
| Char TF-IDF + LR | 0.386 ± 0.032 |
| **Embeddings (all-MiniLM-L6-v2) + LR** | **0.621 ± 0.032** |

- Embeddings improve the score on new merchants by ~60% and win in every fold.
- The pretrained model brings knowledge TF-IDF cannot learn from ~480 merchants: what words
  mean ("festival" ≈ "carnival") and, partly, what brands are.
- Same logistic regression; only the text -> numbers step changed.

In [12]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_emb[train_r.index], train_r["category"])
pred = lr.predict(X_emb[test_r.index])
print("embeddings_lr random split:", round(f1_score(test_r["category"], pred, average="macro"), 3))

embeddings_lr random split: 0.98


In [13]:
lr_m = LogisticRegression(max_iter=1000)
lr_m.fit(X_emb[train_idx], train_m["category"])
test_m["pred_emb"] = lr_m.predict(X_emb[test_idx])
test_m["correct_emb"] = test_m["pred_emb"] == test_m["category"]

compare = test_m.groupby("merchant")[["correct", "correct_emb"]].mean().round(2)
print("Fixed by embeddings:\n", compare[(compare["correct"] < 0.5) & (compare["correct_emb"] > 0.5)])
print("\nBroken by embeddings:\n", compare[(compare["correct"] > 0.5) & (compare["correct_emb"] < 0.5)])
print("\nStill wrong:\n", compare[(compare["correct"] < 0.5) & (compare["correct_emb"] < 0.5)])

Fixed by embeddings:
                         correct  correct_emb
merchant                                    
ambulance                  0.00         0.69
annuity                    0.05         1.00
arcade                     0.00         1.00
bonus                      0.00         0.99
carnival                   0.00         1.00
charter                    0.00         0.96
chase                      0.00         0.98
chiropractor               0.01         1.00
dsl                        0.09         0.94
festival                   0.00         0.97
freelance                  0.02         1.00
fundraising                0.00         1.00
hughesnet                  0.00         0.66
marta                      0.00         0.99
mediacom                   0.01         0.84
medicare                   0.00         0.79
mount sinai                0.00         0.79
netflix                    0.00         0.65
passport                   0.00         1.00
religious organization     0.07  

### What embeddings changed (single merchant split)
- Random split: 0.98 (TF-IDF 0.986), so almost no loss on known merchants.
- Fixed 31 merchants: generic words through meaning (`festival`, `wage`, `passport`, `x-ray`) and
  well-known brands through pretrained knowledge (`chase`, `netflix`, `shell`, `mount sinai`).
- Broke 6 merchants the char model got right (`saks off 5th`, `lord & taylor`, `rally's`),
  so the two approaches are complementary.
- Still wrong for both: lesser-known retail brands (`uniqlo`, `h&m`, `burlington`), hospitals named
  after places/people (`cedars-sinai`, `nyu langone`), acronyms (`bart`, `atf`) and ambiguous names
  (`dollar`, `national`, `refund`).

In [14]:
from scipy.sparse import hstack, csr_matrix

desc, y = clean["transaction_description"], clean["category"]
combo_scores = []

for tr, te in gkf.split(clean, groups=clean["merchant"]):
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True)
    X_tr = hstack([vec.fit_transform(desc.iloc[tr]), csr_matrix(X_emb[tr])]).tocsr()
    X_te = hstack([vec.transform(desc.iloc[te]), csr_matrix(X_emb[te])]).tocsr()

    lr = LogisticRegression(max_iter=1000).fit(X_tr, y.iloc[tr])
    combo_scores.append(f1_score(y.iloc[te], lr.predict(X_te), average="macro"))

combo_scores = np.array(combo_scores)
cv_results["char_tfidf+emb_lr"] = combo_scores
print(f"char_tfidf+emb_lr: {combo_scores.mean():.3f} ± {combo_scores.std():.3f}   folds: {combo_scores.round(3)}")

char_tfidf+emb_lr: 0.667 ± 0.024   folds: [0.683 0.7   0.672 0.636 0.646]


### Merchant split, GroupKFold (5 folds, macro-F1)

| Model | New merchants |
|---|---|
| Dummy | ~0.01 |
| Word TF-IDF + LR | 0.323 ± 0.018 |
| Char TF-IDF + LR | 0.386 ± 0.032 |
| Embeddings (MiniLM) + LR | 0.621 ± 0.032 |
| **Char TF-IDF + embeddings + LR** | **0.667 ± 0.024** |

- Combining both feature types beats each one alone, in every fold: letter chunks catch
  spelling-based matches (`saks off 5th`, `rally's`), embeddings bring word meaning and
  brand knowledge (`festival`, `chase`, `netflix`).
- On known merchants (random split) all TF-IDF/embedding models score ~0.98–0.99.

In [15]:
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True)
X_tr = hstack([vec.fit_transform(train_r["transaction_description"]), csr_matrix(X_emb[train_r.index])]).tocsr()
X_te = hstack([vec.transform(test_r["transaction_description"]), csr_matrix(X_emb[test_r.index])]).tocsr()
lr = LogisticRegression(max_iter=1000).fit(X_tr, train_r["category"])
print("combo random split:", round(f1_score(test_r["category"], lr.predict(X_te), average="macro"), 3))

combo random split: 0.986


In [17]:
import sys; sys.path.append("..")

from ml.features import SentenceEmbedder

emb = SentenceEmbedder()
print(emb.fit_transform(["Netflix", "Netflix", "Uber"]).shape)   # (3, 384)
print(emb.get_params())   # {'batch_size': 256, 'model_name': 'all-MiniLM-L6-v2'}

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(3, 384)
{'batch_size': 256, 'model_name': 'all-MiniLM-L6-v2'}


## Final Model

In [18]:
import joblib
from sklearn.pipeline import make_union

final_model = make_pipeline(
    make_union(
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True),
        SentenceEmbedder(),
    ),
    LogisticRegression(max_iter=1000),
)
final_model.fit(clean["transaction_description"], clean["category"])

model_path = Path("../artifacts/category_model.joblib")
model_path.parent.mkdir(exist_ok=True)
joblib.dump(final_model, model_path)
print(f"Saved: {model_path.stat().st_size / 1e6:.1f} MB")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Saved: 1.9 MB


In [19]:
loaded = joblib.load(model_path)

examples = ["Walgreens #8780", "Joe's Pizza Bar", "Netflix", "Frontier", "Uniqlo",
            "dinner at italian restaurant", "monthly rent"]
proba = loaded.predict_proba(examples)
classes = loaded.classes_

for text, p in zip(examples, proba):
    top2 = p.argsort()[::-1][:2]
    print(f"{text:30} -> " + " | ".join(f"{classes[i]} {p[i]:.0%}" for i in top2))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Walgreens #8780                -> Shopping & Retail 51% | Healthcare & Medical 49%
Joe's Pizza Bar                -> Food & Dining 100% | Transportation 0%
Netflix                        -> Entertainment & Recreation 100% | Utilities & Services 0%
Frontier                       -> Transportation 52% | Utilities & Services 48%
Uniqlo                         -> Shopping & Retail 100% | Transportation 0%
dinner at italian restaurant   -> Food & Dining 55% | Financial Services 22%
monthly rent                   -> Income 98% | Financial Services 1%


In [20]:
phrases = {
    "coffee with friends": "Food & Dining",
    "weekly groceries": "Food & Dining",
    "uber to the airport": "Transportation",
    "train ticket to london": "Transportation",
    "new running shoes": "Shopping & Retail",
    "electricity bill": "Utilities & Services",
    "phone bill": "Utilities & Services",
    "dentist appointment": "Healthcare & Medical",
    "cinema tickets": "Entertainment & Recreation",
    "gym membership": "Entertainment & Recreation",
    "salary october": "Income",
    "credit card fee": "Financial Services",
    "donation to red cross": "Charity & Donations",
    "passport renewal": "Government & Legal",
    "monthly rent": "Utilities & Services",
}

pred = loaded.predict(list(phrases))
conf = loaded.predict_proba(list(phrases)).max(axis=1)
check = pd.DataFrame({"expected": phrases.values(), "pred": pred, "conf": conf.round(2)},
                     index=phrases.keys())
check["ok"] = check["expected"] == check["pred"]
print(check)
print(f"\nCorrect: {check['ok'].sum()}/{len(check)}")

                                          expected  \
coffee with friends                  Food & Dining   
weekly groceries                     Food & Dining   
uber to the airport                 Transportation   
train ticket to london              Transportation   
new running shoes                Shopping & Retail   
electricity bill              Utilities & Services   
phone bill                    Utilities & Services   
dentist appointment           Healthcare & Medical   
cinema tickets          Entertainment & Recreation   
gym membership          Entertainment & Recreation   
salary october                              Income   
credit card fee                 Financial Services   
donation to red cross          Charity & Donations   
passport renewal                Government & Legal   
monthly rent                  Utilities & Services   

                                              pred  conf     ok  
coffee with friends            Charity & Donations  0.56  False  
wee